# Game_Surf NPC Training - Google Colab

Train LoRA adapters for Unity NPC characters using Google Colab GPUs.

## Output
- **LoRA adapter** (`.safetensors`) - For PEFT inference
- **GGUF** (`.gguf`) - For Unity llama.cpp runtime

---

## 1. Setup Environment

In [3]:
# @title ## 1.1 Install Dependencies
!pip install unsloth[torch] transformers datasets trl accelerate bitsandbytes peft --quiet

In [4]:
# @title ## 1.2 HuggingFace Login
from huggingface_hub import notebook_login
notebook_login()

In [5]:
# @title ## 1.3 Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [6]:
# @title ## 1.4 GPU Detection
import torch

def get_gpu_info():
    """Get GPU info for configuration."""
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        return name, total
    return "CPU", 0

GPU_NAME, GPU_VRAM = get_gpu_info()
print(f"✓ GPU: {GPU_NAME}")
print(f"✓ VRAM: {GPU_VRAM:.1f}GB")

In [7]:
# @title ## 1.5 Verify Dataset Path
import os

# List available datasets in your Drive
base_path = '/content/drive/My Drive/game_surf/datasets/processed/'
print(f"Checking: {base_path}")
if os.path.exists(base_path):
    print("Available datasets:")
    for d in sorted(os.listdir(base_path)):
        print(f"  - {d}")
else:
    print("Folder not found! Create in Google Drive:")
    print("  game_surf/datasets/processed/<your_npc>/train.jsonl")

---

## 2. Configuration

In [8]:
# @title ## 2.1 NPC & Dataset Settings
# ==== EDIT THESE FOR YOUR NPC ====
# Upload your dataset to Google Drive first at:
# /content/drive/My Drive/game_surf/datasets/processed/<npc>/train.jsonl

NPC_KEY = 'movies_instructor'  # @param {type:"string"}
DATASET_PATH = '/content/drive/My Drive/game_surf/datasets/processed/movies_instructor_dataset/train.jsonl'  # @param {type:"string"}
VAL_PATH = '/content/drive/My Drive/game_surf/datasets/processed/movies_instructor_dataset/validation.jsonl'  # @param {type:"string"}
OUTPUT_DIR = f'/content/drive/My Drive/game_surf/exports/npc_models/{NPC_KEY}/'  # @param {type:"string"}

print(f"NPC: {NPC_KEY}")
print(f"Dataset: {DATASET_PATH}")
print(f"Output: {OUTPUT_DIR}")


In [9]:
# @title ## 2.2 Model Settings (FORCE 8B)
# ==== FORCE 8B MODEL ON T4 ====
MODEL_NAME = 'unsloth/Llama-3.1-8B-Instruct'  # @param {type:"string"}
MAX_SEQ_LENGTH = 1024  # Reduced to help with T4 memory
LOAD_IN_4BIT = True   # 4-bit to fit in T4 memory
DTYPE = torch.float16

print(f"Model: {MODEL_NAME}")
print(f"Max Seq Length: {MAX_SEQ_LENGTH}")
print(f"4-bit: {LOAD_IN_4BIT}")

In [10]:
# @title ## 2.3 Training Hyperparameters
# LoRA - Rank-Stabilized for persona capture
LORA_R = 32  # @param {type:"integer"}
LORA_ALPHA = 64  # @param {type:"integer"} (2x r for stable scaling)
LORA_DROPOUT = 0.05
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

# Training
EPOCHS = 3  # @param {type:"integer"}
BATCH_SIZE = 1
GRAD_ACCUM = 16  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
WARMUP_STEPS = 20
WEIGHT_DECAY = 0.03
NEFTUNE_ALPHA = 7.5  # Better generalization

# Scheduler
LR_SCHEDULER = 'cosine'
LR_MIN_RATIO = 0.05

# Checkpointing
SAVE_STEPS = 50
EVAL_STEPS = 50
SAVE_TOTAL_LIMIT = 2
LOGGING_STEPS = 10

# Data
VAL_SPLIT = 0.1
SEED = 3407

print(f"LoRA: r={LORA_R}, alpha={LORA_ALPHA}")
print(f"Epochs: {EPOCHS}, LR: {LEARNING_RATE}, Batch: {BATCH_SIZE}")

---

## 3. Load Dataset

In [11]:
# @title ## 3.1 Load Training Data
from datasets import load_dataset

# Use path directly from config
print(f"Loading: {DATASET_PATH}")
train_ds = load_dataset('json', data_files=DATASET_PATH, split='train')


In [12]:
# @title ## 3.2 Load Validation Data (if available)
val_ds = None
if VAL_PATH and VAL_PATH != '':
    try:
        val_path = VAL_PATH  # Use path directly
        val_ds = load_dataset('json', data_files=val_path, split='train')
        print(f"✓ Loaded {len(val_ds)} validation samples")
    except Exception as e:
        print(f"Validation file not found: {e}")

# Fallback: split from train if no val file
if val_ds is None:
    split = train_ds.train_test_split(test_size=VAL_SPLIT, seed=SEED)
    train_ds = split['train']
    val_ds = split['test']
    print(f"⚠ Using split: Train {len(train_ds)}, Val {len(val_ds)}")
else:
    print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

---

## 4. Load Model

In [13]:
# @title ## 4.1 Load Model & Tokenizer
import torch
from unsloth import FastLanguageModel, get_chat_template, is_bfloat16_supported

# Clear CUDA cache
torch.cuda.empty_cache()

print(f"Loading {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    device_map='auto',
)

print(f"✓ Model loaded: {MODEL_NAME}")

In [14]:
# @title ## 4.2 Apply Chat Template
# Apply llama-3 chat template (required for Llama 3.x)
tokenizer = get_chat_template(tokenizer, chat_template='llama-3')

# Ensure pad token is set
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"✓ Tokenizer: {tokenizer.__class__.__name__}")
print(f"✓ Vocab size: {tokenizer.vocab_size}")
print(f"✓ Pad token: {tokenizer.pad_token_id}")

---

## 5. Setup LoRA

In [15]:
# @title ## 5.1 Configure LoRA Adapter
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
    use_rslora=True,  # Rank-stabilized LoRA
    loftq_config=None,
)

model.print_trainable_parameters()

---

## 6. Format Data

In [16]:
# @title ## 6.1 Format to ChatML
def format_to_chatml(examples, tokenizer):
    """Format examples to ChatML for training."""
    texts = []
    for messages in examples['messages']:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {'text': texts}


train_ds = train_ds.map(
    lambda x: format_to_chatml(x, tokenizer),
    batched=True,
    remove_columns=train_ds.column_names,
    desc='Formatting train',
)

val_ds = val_ds.map(
    lambda x: format_to_chatml(x, tokenizer),
    batched=True,
    remove_columns=val_ds.column_names,
    desc='Formatting val',
)

print(f"✓ Formatted {len(train_ds)} train, {len(val_ds)} val")

---

## 7. Training

In [17]:
# @title ## 7.1 Configure Trainer
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
from unsloth.chat_templates import train_on_responses_only
import os

# Create output directory
output_dir = OUTPUT_DIR.replace('My Drive', 'MyDrive').replace('/content/drive/', '/content/')
os.makedirs(output_dir, exist_ok=True)

# SFT Configuration
sft_config = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=WARMUP_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=LOGGING_STEPS,
    optim='paged_adamw_8bit',
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type=LR_SCHEDULER,
    seed=SEED,
    report_to='none',
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    max_grad_norm=1.0,
    # Checkpointing
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    # Evaluation
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16_full_eval=True,
    per_device_eval_batch_size=2,
    eval_accumulation_steps=4,
)

# Data collator
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    max_length=MAX_SEQ_LENGTH,
    pad_to_multiple_of=16,
    return_tensors='pt',
)

# Create trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=collator,
    args=sft_config,
    neftune_noise_alpha=NEFTUNE_ALPHA,
)

# CRITICAL: Train ONLY on assistant responses (not system/user)
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|start_header_id|>user<|end_header_id|>\n\n',
    response_part='<|start_header_id|>assistant<|end_header_id|>\n\n',
)

print(f"✓ Trainer ready: {output_dir}")

In [18]:
# @title ## 7.2 Start Training
print("=" * 50)
print("Starting training...")
print("=" * 50)

trainer.train()

print("=" * 50)
print("Training complete!")
print("=" * 50)

---

## 8. Save Artifacts

In [19]:
# @title ## 8.1 Save LoRA Adapter
import json

# Save adapter
trainer.save_model(output_dir)
print(f"✓ LoRA adapter saved: {output_dir}")

# Save run config
config = {
    'model_name': MODEL_NAME,
    'npc_key': NPC_KEY,
    'max_seq_length': MAX_SEQ_LENGTH,
    'lora_r': LORA_R,
    'lora_alpha': LORA_ALPHA,
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'batch_size': BATCH_SIZE,
    'gradient_accumulation_steps': GRAD_ACCUM,
    'gpu': GPU_NAME,
    'vram_gb': GPU_VRAM,
}

config_path = f"{output_dir}/run_config.json"
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"✓ Config saved: {config_path}")

In [20]:
# @title ## 8.2 Export to GGUF
import os
from pathlib import Path

gguf_dir = f"{output_dir}/gguf"
os.makedirs(gguf_dir, exist_ok=True)

print("Exporting to GGUF (this may take a few minutes)...")

try:
    # Try Q4_K_M first (good quality/size balance)
    model.save_pretrained_gguf(
        gguf_dir,
        tokenizer,
        quantization_method='q4_k_m',
        maximum_memory_usage=0.7,
    )
    print(f"✓ GGUF saved: {gguf_dir}")
    
    # Find GGUF file
    gguf_files = list(Path(gguf_dir).glob('*.gguf'))
    if gguf_files:
        print(f"File: {gguf_files[0].name}")
    
except Exception as e:
    print(f"⚠ GGUF export failed: {e}")
    print("LoRA adapter saved - convert locally with convert_lora_to_gguf.py")

In [21]:
# @title ## 8.3 Create Unity Manifest
import json
from datetime import datetime
from pathlib import Path

output_path = Path(output_dir)
gguf_files = list(output_path.rglob('*.gguf'))
adapter_files = list(output_path.rglob('*.safetensors'))

manifest = {
    'npc_key': NPC_KEY,
    'model_name': MODEL_NAME,
    'adapter_type': 'lora',
    'adapter_format': 'gguf' if gguf_files else 'peft',
    'gguf_path': str(gguf_files[0].relative_to(output_path)) if gguf_files else None,
    'max_seq_length': MAX_SEQ_LENGTH,
    'lora_r': LORA_R,
    'training_config': {
        'epochs': EPOCHS,
        'learning_rate': LEARNING_RATE,
        'batch_size': BATCH_SIZE,
    },
    'gpu': GPU_NAME,
    'created': datetime.now().isoformat(),
}

manifest_path = f"{output_dir}/model_manifest.json"
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f"✓ Manifest saved: {manifest_path}")
print(json.dumps(manifest, indent=2))

---

## 9. Summary

In [24]:
# @title ## 9.1 Show Output Files
from pathlib import Path

output_path = Path(output_dir)
print("=" * 50)
print("OUTPUT FILES")
print("=" * 50)

print(f"\nOutput directory: {output_dir}\n")

for file in sorted(output_path.rglob('*')):
    if file.is_file():
        size_mb = file.stat().st_size / 1e6
        print(f"  {file.name}: {size_mb:.1f}MB")

print("\n" + "=" * 50)
print("✓ Training complete!")
print("=" * 50)

In [25]:
# @title ## 9.2 Verify GGUF
from pathlib import Path

output_path = Path(output_dir)
gguf_files = list(output_path.rglob('*.gguf'))

if gguf_files:
    print("✓ GGUF file available for Unity/llama.cpp")
    print(f"  Path: {gguf_files[0]}")
else:
    print("⚠ No GGUF - use convert_lora_to_gguf.py locally")

In [27]:
import os
from pathlib import Path
output_dir = Path('/content/drive/My Drive/game_surf/exports/npc_models/movies_instructor/')
print(f"Contents: {output_dir}")
for f in sorted(output_dir.rglob('*')):
    if f.is_file():
        print(f"  {f.name}: {f.stat().st_size/1e6:.1f}MB")

In [28]:
from google.colab import drive
drive.mount('/content/drive')

In [26]:
import shutil
shutil.make_archive('/content/lora_adapter', 'zip', '/content/drive/My Drive/game_surf/exports/npc_models/movies_instructor/lora_adapter')